This notebook audits the data structure and quality before modeling, it builds target series. It also checks missing values (like timestamp, vm, etc.), quality, metadata and targets. And it gives a clear conclusion about the data. "The data is clean", "active_vm_count is constant" and "there is no churn in window". Only the first 110 time steps of the targets are inspected, the last 24 holdout are not looked at.

## 1. Load data

We load the CPU readings into `readings` and the VM metadata into `metadata`.

In [1]:
import logging

import pandas as pd

from cwfc.targets import build_targets

logging.basicConfig(level=logging.INFO)

readings = pd.read_csv("../challenge_data/vm_cpu_readings.csv.gz")
metadata = pd.read_csv("../challenge_data/vm_metadata.csv.gz")

## 2. Structure and missing values

Columns and data types of the readings.

In [2]:
readings.info()

<class 'pandas.DataFrame'>
RangeIndex: 1340000 entries, 0 to 1339999
Data columns (total 5 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   timestamp  1340000 non-null  int64  
 1   vm_id      1340000 non-null  str    
 2   min_cpu    1340000 non-null  float64
 3   max_cpu    1340000 non-null  float64
 4   avg_cpu    1340000 non-null  float64
dtypes: float64(3), int64(1), str(1)
memory usage: 51.1 MB


Missing values in the readings.

In [3]:
readings.isna().sum()

timestamp    0
vm_id        0
min_cpu      0
max_cpu      0
avg_cpu      0
dtype: int64

Columns and data types of the metadata.

In [4]:
metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   vm_id                 10000 non-null  str    
 1   subscription_id       10000 non-null  str    
 2   deployment_id         10000 non-null  str    
 3   timestamp_vm_created  10000 non-null  int64  
 4   timestamp_vm_deleted  10000 non-null  int64  
 5   max_cpu               10000 non-null  float64
 6   avg_cpu               10000 non-null  float64
 7   p95_max_cpu           10000 non-null  float64
 8   vm_category           10000 non-null  str    
 9   virtual_core_bucket   10000 non-null  str    
 10  memory_gb_bucket      10000 non-null  str    
dtypes: float64(3), int64(2), str(6)
memory usage: 859.5 KB


Missing values in the metadata.

In [5]:
metadata.isna().sum()

vm_id                   0
subscription_id         0
deployment_id           0
timestamp_vm_created    0
timestamp_vm_deleted    0
max_cpu                 0
avg_cpu                 0
p95_max_cpu             0
vm_category             0
virtual_core_bucket     0
memory_gb_bucket        0
dtype: int64

**Finding:**
There are no missing values in metadata and readings. Our validation filter won't eliminate any rows.

## 3. Coverage

Number of distinct timestamps and VMs.

In [6]:
readings[["timestamp", "vm_id"]].nunique()

timestamp      134
vm_id        10000
dtype: int64

Gaps between consecutive timestamps: take the unique timestamps, sort them, take the difference to the previous one and count each gap size.

In [7]:
pd.Series(sorted(readings["timestamp"].unique())).diff().value_counts()

300.0    133
Name: count, dtype: int64

Number of readings per VM.

In [8]:
readings.groupby("vm_id").size().value_counts()

134    10000
Name: count, dtype: int64

**Finding:**
Serie has equal intervals and has no space. So, it doesn't have to resampling or gap-filling.  

134 timestamps, 10000 VMs and every step is 300 seconds. Each VM has 134 measurements.

## 4. Quality

Duplicate `(timestamp, vm_id)` pairs.

In [9]:
readings.duplicated(subset=["timestamp", "vm_id"]).sum()

np.int64(0)

Summary of the CPU columns.

In [10]:
readings[["min_cpu", "max_cpu", "avg_cpu"]].describe()

,min_cpu,max_cpu,avg_cpu
count,1.340000e+06,1.340000e+06,1.340000e+06
mean,6.156478e+00,1.834830e+01,9.388101e+00
std,1.175525e+01,2.161651e+01,1.426601e+01
min,0.000000e+00,1.810795e-02,7.158922e-03
25%,1.487979e+00,4.417642e+00,2.477060e+00
50%,3.227996e+00,9.124204e+00,5.105133e+00
75%,5.331087e+00,2.362418e+01,9.616715e+00
max,9.928158e+01,9.970286e+01,9.948803e+01


**Finding:**
There are no duplicate (timestamp, vm_id) rows, and all CPU values are within [0, 100]. So the range check doesn’t eliminate any rows.

## 5. Metadata

Summary of the numeric metadata columns.

In [11]:
metadata.describe()

,timestamp_vm_created,timestamp_vm_deleted,max_cpu,avg_cpu,p95_max_cpu
count,10000.0,1.000000e+04,10000.000000,10000.000000,10000.000000
mean,0.0,2.289626e+06,86.492232,9.338249,36.365331
std,0.0,7.081008e+05,18.760321,11.253499,29.437773
min,0.0,4.050000e+04,0.028883,0.008060,0.025152
25%,0.0,2.591400e+06,82.299583,3.046473,10.591030
50%,0.0,2.591400e+06,95.423559,5.859044,23.975787
75%,0.0,2.591400e+06,98.381921,10.532403,62.029330
max,0.0,2.591400e+06,100.000000,99.199464,99.519122


VM creation times.

In [12]:
metadata["timestamp_vm_created"].value_counts()

timestamp_vm_created
0    10000
Name: count, dtype: int64

VM deletion times: summary and counts.

In [13]:
metadata["timestamp_vm_deleted"].describe()

count    1.000000e+04
mean     2.289626e+06
std      7.081008e+05
min      4.050000e+04
25%      2.591400e+06
50%      2.591400e+06
75%      2.591400e+06
max      2.591400e+06
Name: timestamp_vm_deleted, dtype: float64

In [14]:
metadata["timestamp_vm_deleted"].value_counts()

timestamp_vm_deleted
2591400    8213
736500       50
1537200      42
743100       37
352200       20
           ... 
2237700       1
624000        1
1017600       1
2199000       1
245700        1
Name: count, Length: 1071, dtype: int64

VM categories.

In [15]:
metadata["vm_category"].value_counts()

vm_category
Delay-insensitive    6180
Interactive          3091
Unknown               729
Name: count, dtype: int64

**Finding:**
Finding: No VM is deleted within the window (earliest deletion 40,500 s, last reading 39,900 s), and 82% of VMs live until the end of the trace. There is no churn, so active_vm_count stays constant; the VMs were likely selected for this, which limits how well capacity findings generalise.

## 6. Targets (training window only)

The targets are built with `build_targets` from `src/cwfc/targets.py`. Only the training window (all steps except the last 24) is shown; the last 24 steps are the holdout.

In [16]:
targets = build_targets(readings)
train_targets = targets.iloc[:-24]
train_targets.head()

INFO:cwfc.targets:Dropped 0 of 1340000 rows


,timestamp,active_vm_count,aggregate_cpu_utilization
0,0,10000,9.396898
1,300,10000,9.353775
2,600,10000,9.454383
3,900,10000,9.224419
4,1200,10000,9.143503


In [17]:
train_targets["active_vm_count"].unique()

array([10000])

**Finding:** 
The validity filter drops 0 rows, and active_vm_count is constant at 10,000 in the training window. Persistence is exact for this target, so normalised metrics would divide by zero.